In [ ]:
# ============================================================
# CELL 1
# IMPORTS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import time

import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder

from numpy.linalg import inv

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 2
# DATASET SETUP
# ============================================================

DATASET_ROOT = "/content/drive/MyDrive/Potato"

SAVE_DIR = os.path.join(
    DATASET_ROOT,
    "run"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

RANDOM_STATE = 42

IMAGE_SIZE = 224

CLASS_NAMES = sorted([

    folder

    for folder in os.listdir(DATASET_ROOT)

    if os.path.isdir(
        os.path.join(DATASET_ROOT, folder)
    )

    and folder != "run"

])

print("="*60)
print("CLASSES")
print("="*60)

print(CLASS_NAMES)

# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

rows = []

for cls in CLASS_NAMES:

    class_dir = os.path.join(
        DATASET_ROOT,
        cls
    )

    for file in os.listdir(class_dir):

        if file.lower().endswith((
            ".jpg",
            ".jpeg",
            ".png"
        )):

            rows.append({

                "Image_ID": file,

                "image_path":
                    os.path.join(class_dir, file),

                "class": cls

            })

df = pd.DataFrame(rows)

print()

print("Total Images :", len(df))

print()

print(df["class"].value_counts())

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["class"]
)

NUM_CLASSES = len(
    label_encoder.classes_
)

print()

print("Label Mapping")

for i, cls in enumerate(
    label_encoder.classes_
):

    print(i, "->", cls)

# ------------------------------------------------------------
# Train / Validation Split
# ------------------------------------------------------------

df_train, df_val = train_test_split(

    df,

    test_size=0.20,

    stratify=df["label"],

    random_state=RANDOM_STATE

)

df_train = df_train.reset_index(
    drop=True
)

df_val = df_val.reset_index(
    drop=True
)

print()

print("="*60)
print("TRAIN SPLIT")
print("="*60)

print(df_train["class"].value_counts())

print()

print("="*60)
print("VALIDATION SPLIT")
print("="*60)

print(df_val["class"].value_counts())

# ------------------------------------------------------------
# Save splits
# ------------------------------------------------------------

TRAIN_SPLIT_PATH = os.path.join(

    SAVE_DIR,

    "train_split.csv"

)

VAL_SPLIT_PATH = os.path.join(

    SAVE_DIR,

    "validation_split.csv"

)

df_train.to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)

df_val.to_csv(
    VAL_SPLIT_PATH,
    index=False
)

print()

print("Saved")

print(TRAIN_SPLIT_PATH)

print(VAL_SPLIT_PATH)

# ============================================================
# LOAD PRETRAINED MOBILENET
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

weights = MobileNet_V3_Small_Weights.DEFAULT

model = mobilenet_v3_small(
    weights=weights
)

print()

print("="*60)
print("PRETRAINED MOBILENET LOADED")
print("="*60)

print(model)

# ============================================================
# CONVERT TO FEATURE EXTRACTOR
# ============================================================

feature_extractor = nn.Sequential(

    model.features,

    model.avgpool,

    nn.Flatten()

).to(device)

feature_extractor.eval()

dummy = torch.randn(
    1,
    3,
    224,
    224
).to(device)

with torch.no_grad():

    output = feature_extractor(dummy)

print()

print("="*60)
print("FEATURE EXTRACTOR")
print("="*60)

print("Feature Shape :", output.shape)

CLASSES
['Early_Blight', 'Healthy', 'Late_Blight']

Total Images : 2152

class
Early_Blight    1000
Late_Blight     1000
Healthy          152
Name: count, dtype: int64

Label Mapping
0 -> Early_Blight
1 -> Healthy
2 -> Late_Blight

TRAIN SPLIT
class
Early_Blight    800
Late_Blight     800
Healthy         121
Name: count, dtype: int64

VALIDATION SPLIT
class
Late_Blight     200
Early_Blight    200
Healthy          31
Name: count, dtype: int64

Saved
/content/drive/MyDrive/Potato/run/train_split.csv
/content/drive/MyDrive/Potato/run/validation_split.csv
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 64.0MB/s]



PRETRAINED MOBILENET LOADED
MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d

In [ ]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

CLASS_NAMES_PATH = os.path.join(
    SAVE_DIR,
    "class_names.json"
)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(
        CLASS_NAMES,
        f,
        indent=2
    )

print("Class names saved to:")
print(CLASS_NAMES_PATH)

print()
print("Classes:")
print(CLASS_NAMES)

Class names saved to:
/content/drive/MyDrive/Potato/run/class_names.json

Classes:
['Early_Blight', 'Healthy', 'Late_Blight']


In [ ]:
train_ids = set(df_train["Image_ID"])
val_ids = set(df_val["Image_ID"])

overlap = train_ids.intersection(val_ids)

print("Overlap:", len(overlap))

Overlap: 0


In [ ]:
# ============================================================
# CELL 3
# EXTRACT MOBILENET FEATURES
# ============================================================

TRAIN_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "train_features.csv"
)

VAL_FEATURE_PATH = os.path.join(
    SAVE_DIR,
    "validation_features.csv"
)

# ------------------------------------------------------------
# Image Transform
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# Load Existing Features
# ------------------------------------------------------------

if (
    os.path.exists(TRAIN_FEATURE_PATH)
    and
    os.path.exists(VAL_FEATURE_PATH)
):

    print("=" * 60)
    print("FEATURE FILES FOUND")
    print("=" * 60)

    extracted_features_train = pd.read_csv(
        TRAIN_FEATURE_PATH
    )

    extracted_features_val = pd.read_csv(
        VAL_FEATURE_PATH
    )

    print(
        extracted_features_train.shape
    )

    print(
        extracted_features_val.shape
    )

else:

    # --------------------------------------------------------
    # Feature Extraction Function
    # --------------------------------------------------------

    def extract_features(dataframe):

        rows = []

        feature_extractor.eval()

        with torch.no_grad():

            for _, row in tqdm(
                dataframe.iterrows(),
                total=len(dataframe)
            ):

                image = Image.open(
                    row["image_path"]
                ).convert("RGB")

                image = transform(image)

                image = image.unsqueeze(0).to(device)

                features = feature_extractor(
                    image
                )

                features = (
                    features
                    .cpu()
                    .numpy()
                    .flatten()
                )

                rows.append([

                    row["Image_ID"],

                    row["class"],

                    *features

                ])

        columns = (

            ["Image_ID", "class"]

            +

            [
                f"feat_{i}"
                for i in range(576)
            ]

        )

        return pd.DataFrame(
            rows,
            columns=columns
        )

    # --------------------------------------------------------
    # Training Features
    # --------------------------------------------------------

    print()

    print("=" * 60)
    print("TRAIN FEATURES")
    print("=" * 60)

    extracted_features_train = extract_features(
        df_train
    )

    # --------------------------------------------------------
    # Validation Features
    # --------------------------------------------------------

    print()

    print("=" * 60)
    print("VALIDATION FEATURES")
    print("=" * 60)

    extracted_features_val = extract_features(
        df_val
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    extracted_features_train.to_csv(

        TRAIN_FEATURE_PATH,

        index=False

    )

    extracted_features_val.to_csv(

        VAL_FEATURE_PATH,

        index=False

    )

    print()

    print("Saved")

    print(TRAIN_FEATURE_PATH)

    print(VAL_FEATURE_PATH)

print()

print(extracted_features_train.head())

print()

print(extracted_features_train.shape)

print(extracted_features_val.shape)

FEATURE FILES FOUND
(1721, 578)
(431, 578)

                                            Image_ID         class    feat_0  \
0  c5464947-d6f8-4d5e-9cad-009db19adf67___RS_Earl...  Early_Blight  0.095066   
1  1d466431-007d-4b3b-bd45-b09f1a6f7bad___RS_Earl...  Early_Blight  0.044866   
2  37075adc-dd51-4a5b-8a6c-0649cb98516b___RS_LB 2...   Late_Blight -0.074745   
3  fc748621-bd56-4012-a617-9e30df58ecfc___RS_LB 4...   Late_Blight  0.031745   
4  a1abff90-34e5-464d-b4d3-0c09f5924900___RS_HL 5...       Healthy -0.091331   

     feat_1    feat_2    feat_3    feat_4    feat_5    feat_6    feat_7  ...  \
0 -0.093432  0.006293 -0.102639  0.434751  0.531164  0.067001  0.053508  ...   
1 -0.042892  0.468777  0.914444 -0.009984  0.687378  0.120632  0.255327  ...   
2 -0.133295  0.054801  0.356272  0.567160  0.475008  0.415748  0.409413  ...   
3 -0.103367  1.026330  1.172394  0.186441  0.619173  0.327964  0.430753  ...   
4 -0.087900  0.370226 -0.077096  0.213160  0.382772  0.097874 -0.120869  ..

In [ ]:
# ============================================================
# CELL 4
# TRAIN JACOBI-DMR
# ============================================================

import time

time_jacobi_start = time.time()

# ------------------------------------------------------------
# Prepare Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_train = extracted_features_train["class"]

X_train = X_train.values

# ------------------------------------------------------------
# One-Hot Encoding
# ------------------------------------------------------------

y_one_hot = pd.get_dummies(
    y_train
)

# ------------------------------------------------------------
# Jacobi Hyperparameters
# ------------------------------------------------------------

a = b = 1 / len(X_train)

k = 1

# ------------------------------------------------------------
# Train One Classifier Per Class
# ------------------------------------------------------------

betas = {}

for cls in y_one_hot.columns:

    eta = np.log(

        (y_one_hot[cls] + a)

        /

        (1 + k * b)

    )

    beta = inv(

        X_train.T @ X_train

    ) @ (

        X_train.T @ eta

    )

    betas[cls] = beta

# ------------------------------------------------------------
# Prepare Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

X_val = X_val.values

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

lambda_scores = {}

for cls in betas:

    lambda_scores[cls] = np.exp(
        X_val @ betas[cls]
    )

lambda_df = pd.DataFrame(
    lambda_scores
)

y_pred = lambda_df.idxmax(
    axis=1
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

time_jacobi_seconds = (
    time.time() -
    time_jacobi_start
)

acc_jacobi = accuracy_score(
    y_true,
    y_pred
)

cm_jacobi = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("JACOBI-DMR")
print("=" * 60)

print(
    f"Accuracy: {acc_jacobi:.4f}"
)

print(
    f"Time: {time_jacobi_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_jacobi)

# ------------------------------------------------------------
# Save Coefficients
# ------------------------------------------------------------

JACOBI_PATH = os.path.join(
    SAVE_DIR,
    "jacobi_coefficients.json"
)

with open(
    JACOBI_PATH,
    "w"
) as f:

    json.dump(
        {
            cls: betas[cls].tolist()
            for cls in betas
        },
        f
    )

print()

print("Jacobi coefficients saved to:")

print(JACOBI_PATH)


JACOBI-DMR
Accuracy: 0.9861
Time: 0.1236s

              precision    recall  f1-score   support

Early_Blight       0.98      1.00      0.99       200
     Healthy       0.94      1.00      0.97        31
 Late_Blight       1.00      0.97      0.98       200

    accuracy                           0.99       431
   macro avg       0.97      0.99      0.98       431
weighted avg       0.99      0.99      0.99       431


Confusion Matrix:

[[200   0   0]
 [  0  31   0]
 [  4   2 194]]

Jacobi coefficients saved to:
/content/drive/MyDrive/Potato/run/jacobi_coefficients.json


In [ ]:
# ============================================================
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier

import time

time_rf_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_rf = rf_model.predict(
    X_val
)

time_rf_seconds = (
    time.time() -
    time_rf_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_rf = accuracy_score(
    y_true,
    y_pred_rf
)

cm_rf = confusion_matrix(
    y_true,
    y_pred_rf,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("RANDOM FOREST")
print("=" * 60)

print(
    f"Accuracy: {acc_rf:.4f}"
)

print(
    f"Time: {time_rf_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_rf,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_rf)


RANDOM FOREST
Accuracy: 0.9791
Time: 3.0721s

              precision    recall  f1-score   support

Early_Blight       0.99      0.98      0.99       200
     Healthy       1.00      0.87      0.93        31
 Late_Blight       0.96      0.99      0.98       200

    accuracy                           0.98       431
   macro avg       0.99      0.95      0.97       431
weighted avg       0.98      0.98      0.98       431


Confusion Matrix:

[[196   0   4]
 [  0  27   4]
 [  1   0 199]]


In [ ]:
# ============================================================
# SUPPORT VECTOR MACHINE (SVM)
# ============================================================

from sklearn.svm import SVC

import time

time_svm_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=42
)

svm_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_svm = svm_model.predict(
    X_val
)

time_svm_seconds = (
    time.time() -
    time_svm_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_svm = accuracy_score(
    y_true,
    y_pred_svm
)

cm_svm = confusion_matrix(
    y_true,
    y_pred_svm,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("SUPPORT VECTOR MACHINE")
print("=" * 60)

print(
    f"Accuracy: {acc_svm:.4f}"
)

print(
    f"Time: {time_svm_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_svm,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_svm)


SUPPORT VECTOR MACHINE
Accuracy: 0.9884
Time: 0.1445s

              precision    recall  f1-score   support

Early_Blight       1.00      0.99      0.99       200
     Healthy       0.94      0.97      0.95        31
 Late_Blight       0.99      0.99      0.99       200

    accuracy                           0.99       431
   macro avg       0.97      0.98      0.98       431
weighted avg       0.99      0.99      0.99       431


Confusion Matrix:

[[198   0   2]
 [  0  30   1]
 [  0   2 198]]


In [ ]:
# ============================================================
# RIDGE CLASSIFIER
# ============================================================

from sklearn.linear_model import RidgeClassifier

import time

time_ridge_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

ridge_model = RidgeClassifier(
    alpha=1.0,
    random_state=42
)

ridge_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_ridge = ridge_model.predict(
    X_val
)

time_ridge_seconds = (
    time.time() -
    time_ridge_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_ridge = accuracy_score(
    y_true,
    y_pred_ridge
)

cm_ridge = confusion_matrix(
    y_true,
    y_pred_ridge,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("RIDGE CLASSIFIER")
print("=" * 60)

print(
    f"Accuracy: {acc_ridge:.4f}"
)

print(
    f"Time: {time_ridge_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_ridge,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_ridge)


RIDGE CLASSIFIER
Accuracy: 0.9954
Time: 0.1692s

              precision    recall  f1-score   support

Early_Blight       1.00      1.00      1.00       200
     Healthy       0.97      1.00      0.98        31
 Late_Blight       1.00      0.99      0.99       200

    accuracy                           1.00       431
   macro avg       0.99      1.00      0.99       431
weighted avg       1.00      1.00      1.00       431


Confusion Matrix:

[[200   0   0]
 [  0  31   0]
 [  1   1 198]]


In [ ]:
# ============================================================
# LOGISTIC REGRESSION (L1 / LASSO)
# ============================================================

from sklearn.linear_model import LogisticRegression

import time

time_lasso_start = time.time()

# ------------------------------------------------------------
# Training Data
# ------------------------------------------------------------

X_train = extracted_features_train.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

Y_train = extracted_features_train["class"]

# ------------------------------------------------------------
# Validation Data
# ------------------------------------------------------------

X_val = extracted_features_val.drop(
    columns=[
        "Image_ID",
        "class"
    ]
)

y_true = extracted_features_val["class"]

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

lasso_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=1.0,
    max_iter=5000,
    random_state=42
)

lasso_model.fit(
    X_train,
    Y_train
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_lasso = lasso_model.predict(
    X_val
)

time_lasso_seconds = (
    time.time() -
    time_lasso_start
)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

acc_lasso = accuracy_score(
    y_true,
    y_pred_lasso
)

cm_lasso = confusion_matrix(
    y_true,
    y_pred_lasso,
    labels=CLASS_NAMES
)

print()

print("=" * 60)
print("LOGISTIC REGRESSION (L1 / LASSO)")
print("=" * 60)

print(
    f"Accuracy: {acc_lasso:.4f}"
)

print(
    f"Time: {time_lasso_seconds:.4f}s"
)

print()

print(
    classification_report(
        y_true,
        y_pred_lasso,
        labels=CLASS_NAMES
    )
)

print()

print("Confusion Matrix:\n")

print(cm_lasso)


LOGISTIC REGRESSION (L1 / LASSO)
Accuracy: 0.9861
Time: 67.5202s

              precision    recall  f1-score   support

Early_Blight       0.99      0.99      0.99       200
     Healthy       0.94      0.97      0.95        31
 Late_Blight       0.99      0.98      0.98       200

    accuracy                           0.99       431
   macro avg       0.97      0.98      0.98       431
weighted avg       0.99      0.99      0.99       431


Confusion Matrix:

[[199   0   1]
 [  0  30   1]
 [  2   2 196]]


In [ ]:
# ============================================================
# FINAL MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [

        "Jacobi-DMR",
        "Random Forest",
        "Support Vector Machine",
        "Ridge Classifier",
        "Logistic Regression (L1)"

    ],

    "Accuracy": [

        acc_jacobi,
        acc_rf,
        acc_svm,
        acc_ridge,
        acc_lasso

    ],

    "Execution Time (s)": [

        time_jacobi_seconds,
        time_rf_seconds,
        time_svm_seconds,
        time_ridge_seconds,
        time_lasso_seconds

    ]

})

comparison_df["Accuracy"] = comparison_df["Accuracy"].round(4)

comparison_df["Execution Time (s)"] = comparison_df[
    "Execution Time (s)"
].round(4)

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print()

print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(comparison_df)

comparison_path = os.path.join(
    SAVE_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print()

print(f"Comparison saved to:\n{comparison_path}")


MODEL COMPARISON
                      Model  Accuracy  Execution Time (s)
0          Ridge Classifier    0.9954              0.1692
1    Support Vector Machine    0.9884              0.1445
2                Jacobi-DMR    0.9861              0.1236
3  Logistic Regression (L1)    0.9861             67.5202
4             Random Forest    0.9791              3.0721

Comparison saved to:
/content/drive/MyDrive/Potato/run/model_comparison.csv
